# Crear tu propio benchmark: evaluación específica del dominio

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/benchmarks/02-evaluacion-propia.ipynb)

Pipeline completo de evaluación con golden dataset y LLM-as-judge.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
import json
from dataclasses import dataclass, field

client = anthropic.Anthropic()

## 1. Golden dataset de ejemplo

In [ ]:
golden_dataset = [
    {
        'pregunta': '¿Qué es el aprendizaje supervisado?',
        'respuesta_esperada': 'El aprendizaje supervisado es un tipo de ML donde el modelo aprende de ejemplos etiquetados (pares entrada-salida conocidos) para predecir la salida de nuevas entradas.',
        'categoria': 'conceptos_ml',
        'dificultad': 'facil',
    },
    {
        'pregunta': '¿Cuándo usar fine-tuning en lugar de RAG?',
        'respuesta_esperada': 'Fine-tuning cuando necesitas cambiar el comportamiento o estilo del modelo con datos propios. RAG cuando necesitas acceso a información actualizada o muy específica sin modificar el modelo base.',
        'categoria': 'decisiones_arquitectura',
        'dificultad': 'media',
    },
    {
        'pregunta': '¿Qué es el overfitting y cómo evitarlo?',
        'respuesta_esperada': 'Overfitting es cuando un modelo memoriza el entrenamiento y falla en datos nuevos. Se evita con: regularización (L1/L2), dropout, early stopping, más datos, o modelos más simples.',
        'categoria': 'conceptos_ml',
        'dificultad': 'media',
    },
]

print(f'Dataset cargado: {len(golden_dataset)} ejemplos')

## 2. LLM-as-judge

In [ ]:
@dataclass
class ResultadoEval:
    puntuacion: float
    correcto: bool
    razon: str

def llm_judge(pregunta: str, respuesta_esperada: str, respuesta_modelo: str) -> ResultadoEval:
    prompt = f"""Evalúa si la respuesta del modelo es correcta (0-10).

PREGUNTA: {pregunta}
RESPUESTA ESPERADA: {respuesta_esperada}
RESPUESTA DEL MODELO: {respuesta_modelo}

JSON: {{"puntuacion": N, "correcto": true/false, "razon": "..."}}"""

    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=128,
        messages=[{'role': 'user', 'content': prompt}],
    )
    datos = json.loads(response.content[0].text)
    return ResultadoEval(puntuacion=datos['puntuacion']/10, correcto=datos['correcto'], razon=datos['razon'])

## 3. Pipeline completo de evaluación

In [ ]:
def evaluar_modelo(modelo: str, dataset: list) -> dict:
    resultados = []
    for ejemplo in dataset:
        # Generar respuesta del modelo
        resp = client.messages.create(
            model=modelo,
            max_tokens=256,
            messages=[{'role': 'user', 'content': ejemplo['pregunta']}],
        )
        respuesta_modelo = resp.content[0].text
        
        # Evaluar con LLM-as-judge
        eval_r = llm_judge(ejemplo['pregunta'], ejemplo['respuesta_esperada'], respuesta_modelo)
        resultados.append({
            'categoria': ejemplo['categoria'],
            'puntuacion': eval_r.puntuacion,
            'correcto': eval_r.correcto,
        })
    
    accuracy = sum(1 for r in resultados if r['correcto']) / len(resultados)
    score_medio = sum(r['puntuacion'] for r in resultados) / len(resultados)
    return {'modelo': modelo, 'accuracy': accuracy, 'score_medio': score_medio, 'n': len(resultados)}

# Evaluar dos modelos
for modelo in ['claude-sonnet-4-6', 'claude-haiku-4-5-20251001']:
    resultado = evaluar_modelo(modelo, golden_dataset)
    print(f"{resultado['modelo']:<35} Accuracy: {resultado['accuracy']:.0%}  Score: {resultado['score_medio']:.2f}")